## 주문(orders) CRUD - orders, order_items

In [ ]:
import os
import uuid
from datetime import datetime, timezone
from dotenv import load_dotenv
from supabase import create_client, Client

# 환경변수 로드
load_dotenv()

url = os.getenv("SUPABASE_URL")
key = os.getenv("SUPABASE_PUBLISHABLE_KEY")

# Supabase 클라이언트 생성
supabase : Client = create_client(url, key)

In [ ]:
response = supabase.table("orders").select("*").execute()

In [ ]:
class Order_service:
    def __init__(self, supabase):
        self.supabase = supabase
        self.table = "orders"

    # 주문 생성
    def create_order(self, order_data):
        response = (
            supabase.table(self.table)
                    .insert(order_data)
                    .execute()
        )
        # 주문상품 만들기
        return response.data

    # 회원별 주문 목록 조회
    def get_orders(self, user_id : str):
        response = (
            supabase.table(self.table)
                    .select(user_id)
                    .eq("user_id", user_id)
                    .execute()
        )

        return response.data

    # 주문 상세 조회
    def get_order(self, user_id : str, order_id : str):
        response = (
            supabase.table(self.table)
                    .select(user_id, order_id)
                    .eq("user_id", user_id)
                    .eq("id", order_id)
                    .execute()
        )
        return response.data

    # 주문 정보 수정
    def update_order(self, user_id, order_id, order_data):
        response = (
            supabase.table(self.table)
                    .update(order_id)
                    .eq("user_id", user_id)
                    .eq("order_date")
                    
        )

    # 주문 상태 변경
    def update_order_status(self, user_id, order_id, status):
        now = datetime.now(timezone.utc).isoformat()
        response = (
            self.supabase.table(self.table)
                .update({
                    "order_status": status,
                    "modified_at": now,
                })
                .eq("user_id", str(user_id))
                .eq("id", str(order_id))
                .is_("deleted_at", "null")
                .execute()
        )
        return response.data

    # 주문 소프트 삭제
    def delete_order(self, user_id, order_id):
        now = datetime.now(timezone.utc).isoformat()
        response = (
            self.supabase.table(self.table)
                .update({
                    "deleted_at": now,
                    "modified_at": now,
                })
                .eq("user_id", str(user_id))
                .eq("id", str(order_id))
                .is_("deleted_at", "null")
                .execute()
        )
        return response.data